## 0. Data preprocess

##### Since dataset contains many different types of features, require preprocess the data.

In [42]:
# import useful library
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_classif
from sklearn.feature_selection import mutual_info_regression
from sklearn.feature_selection import f_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
import matplotlib.pyplot as plt

In [43]:
# import datasets
train_data = pd.read_csv("../COMP30027_2024_asst2_data/train_dataset.csv")
test_data = pd.read_csv("../COMP30027_2024_asst2_data/test_dataset.csv")


# count CountVectorizer features for director names, actor names, and possibly other text features
cv_path = "../COMP30027_2024_asst2_data/features_countvec/"
cvtrain_director_name = np.load(cv_path + "train_countvec_features_director_name.npy")
cvtrain_actor_1_name = np.load(cv_path + "train_countvec_features_actor_1_name.npy")
cvtrain_actor_2_name = np.load(cv_path + "train_countvec_features_actor_2_name.npy")

cvtest_director_name = np.load(cv_path + "test_countvec_features_director_name.npy")
cvtest_actor_1_name = np.load(cv_path + "test_countvec_features_actor_1_name.npy")
cvtest_actor_2_name = np.load(cv_path + "test_countvec_features_actor_2_name.npy")

# Doc2Vec features for plot keywords, genres, and other text features
d2v_path = "../COMP30027_2024_asst2_data/features_doc2vec/"
d2vtrain_plot = np.load(d2v_path + "train_doc2vec_features_plot_keywords.npy")
d2vtrain_genre = np.load(d2v_path + "train_doc2vec_features_genre.npy")
d2vtest_plot = np.load(d2v_path + "test_doc2vec_features_plot_keywords.npy")
d2vtest_genre = np.load(d2v_path + "test_doc2vec_features_genre.npy")

# FastText embeddings for movie titles
ft_path = "../COMP30027_2024_asst2_data/features_fasttext/"
fttrain_title = np.load(ft_path + "train_fasttext_title_embeddings.npy")
fttest_title = np.load(ft_path + "test_fasttext_title_embeddings.npy")

In [44]:
# handle missing values
missing_counts = train_data.isnull().sum()
print(missing_counts)
missing_row = train_data[train_data['language'].isnull()]
# as the language of a movie called "Labor Day" that the country is USA, good chance English
train_data.loc[2787, 'language'] = 'English'

id                           0
director_name                0
num_critic_for_reviews       0
duration                     0
director_facebook_likes      0
actor_3_facebook_likes       0
actor_2_name                 0
actor_1_facebook_likes       0
gross                        0
genres                       0
actor_1_name                 0
movie_title                  0
num_voted_users              0
cast_total_facebook_likes    0
actor_3_name                 0
facenumber_in_poster         0
plot_keywords                0
num_user_for_reviews         0
language                     1
country                      0
content_rating               0
title_year                   0
actor_2_facebook_likes       0
movie_facebook_likes         0
title_embedding              0
average_degree_centrality    0
imdb_score_binned            0
dtype: int64
        id director_name  num_critic_for_reviews  duration  \
2787  2788    Ron Fricke                     115       102   

      director_facebook_l

In [45]:
def convert_to_dataframe(np_array, prefix, index):
    # convert nparray to df
    col_names = [f"{prefix}_feat_{i}" for i in range(np_array.shape[1])]
    return pd.DataFrame(np_array, columns=col_names, index=index)

In [46]:
df_cvtrain_director_name = convert_to_dataframe(cvtrain_director_name, 'director', train_data.index)
df_cvtrain_actor_1_name = convert_to_dataframe(cvtrain_actor_1_name, 'actor_1', train_data.index)
df_cvtrain_actor_2_name = convert_to_dataframe(cvtrain_actor_2_name, 'actor_2', train_data.index)
train_data = pd.concat([train_data.drop(columns=['director_name','actor_1_name', 'actor_2_name']), df_cvtrain_director_name, df_cvtrain_actor_1_name, df_cvtrain_actor_2_name], axis=1)

In [47]:
df_d2vtrain_plot = convert_to_dataframe(d2vtrain_plot, 'plot_keywords', train_data.index)
df_d2vtrain_genre = convert_to_dataframe(d2vtrain_genre, 'genres', train_data.index)
train_data = pd.concat([train_data.drop(columns=['plot_keywords','genres']), df_d2vtrain_plot, df_d2vtrain_genre], axis=1)

In [49]:
df_fttrain_title = convert_to_dataframe(fttrain_title, 'title', train_data.index)
train_data = pd.concat([train_data.drop(columns=['title_embedding']), df_fttrain_title], axis=1)

In [50]:
encoder = OneHotEncoder(sparse=False, handle_unknown='ignore')

# encoding
columns_to_encode = ['language', 'country', 'content_rating']

encoded_data = encoder.fit_transform(train_data[columns_to_encode])

encoded_df = pd.DataFrame(encoded_data, columns=encoder.get_feature_names_out(columns_to_encode))

# replace
train_data = train_data.drop(columns=columns_to_encode)

train_data = pd.concat([train_data, encoded_df], axis=1)

C:\Users\Steven\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\preprocessing\_encoders.py:972: FutureWarning: `sparse` was renamed to `sparse_output` in version 1.2 and will be removed in 1.4. `sparse_output` is ignored unless you leave `sparse` to its default value.
  warnings.warn(


In [51]:
train_data = train_data.drop(columns=['movie_title', 'actor_3_name'])

In [52]:
train_data.to_csv("../COMP30027_2024_asst2_data/train_high_dim.csv", index=False)

In [ ]:
# too much dimension, no useful
# need to feature engineering



In [4]:
# generate a simple numerical dataset for baseline models
# starting by remove all text column
# for non numerical feature, convert to numerical
le = LabelEncoder()
train_data['language'] = le.fit_transform(train_data['language'])
train_data['country'] = le.fit_transform(train_data['country'])
train_data['content_rating'] = le.fit_transform(train_data['content_rating'])

test_data['language'] = le.fit_transform(test_data['language'])
test_data['country'] = le.fit_transform(test_data['country'])
test_data['content_rating'] = le.fit_transform(test_data['content_rating'])

In [7]:
# as those columns contain quite a lot unique values, and they are text based which is hard to process
# generate a train set that just discard them
# by logic, movie genre and plot_keyword should not interfere with rating, test the accuracy if drop them
columns_to_drop = ['director_name', 'actor_1_name', 'actor_2_name', 'actor_3_name', 'movie_title', 'genres', 'plot_keywords', 'title_embedding']
train_data_no_text = train_data.drop(columns=columns_to_drop)
test_data_no_text = test_data.drop(columns=columns_to_drop)

train_data_no_text.to_csv("../COMP30027_2024_asst2_data/train_dataset_no_text.csv", index=False)
test_data_no_text.to_csv("../COMP30027_2024_asst2_data/test_dataset_no_text.csv", index=False)

In [8]:
# Selecting numerical columns except 'id' and the categorical label itself
numerical_cols = train_data_no_text.columns.drop(['id', 'imdb_score_binned'])
X = train_data_no_text[numerical_cols]
y = train_data_no_text['imdb_score_binned']  # This is the categorical label

# Perform ANOVA F-test
f_values, p_values = f_classif(X, y)

# Creating a DataFrame to display the F-values and p-values neatly
results = pd.DataFrame({
    'Feature': numerical_cols,
    'F-value': f_values,
    'p-value': p_values
}).sort_values(by='p-value')

# Display results
print(results)

                      Feature     F-value        p-value
6             num_voted_users  530.075648   0.000000e+00
9        num_user_for_reviews  151.709849  2.505394e-118
1                    duration  103.584817   9.685395e-83
15       movie_facebook_likes   90.151026   1.847476e-72
0      num_critic_for_reviews   89.122897   1.147663e-71
5                       gross   42.651752   7.601010e-35
2     director_facebook_likes   33.174868   4.109599e-27
13                 title_year   27.466067   2.005347e-22
16  average_degree_centrality   24.897300   2.616387e-20
10                   language   15.712583   9.895125e-13
7   cast_total_facebook_likes    7.431214   5.935612e-06
12             content_rating    6.855254   1.724263e-05
14     actor_2_facebook_likes    6.170993   6.078697e-05
4      actor_1_facebook_likes    5.728439   1.366543e-04
8        facenumber_in_poster    5.323933   2.854103e-04
11                    country    5.018093   4.966454e-04
3      actor_3_facebook_likes  